# WeaveGuard AI — Étape B : construction du dataset final (v1)
Décisions validées après l'audit :
- **`textile_defects` = seule source** pour YOLO (BDFD9 et hole_detection écartés).
- **Split par groupes** (photo source + quasi-doublons) pour éviter la fuite de données, stratifié par classe, graine fixe.
- Les images **sans boîte** sont exclues (on ne suppose pas qu'elles sont normales) et listées pour revue manuelle.
- **Recadrages sans défaut** extraits *dans chaque partition séparément* (train / val / test) pour PatchCore.

Sorties : dataset local `/content/weaveguard_dataset_v1/` + zip sur Drive (`WeaveGuard/datasets/`) + `dataset_stats.json` à me coller dans le chat.
Lecture seule sur `textile_defects` : rien n'est modifié dans tes données d'origine.

In [ ]:
# 1) Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Configuration
from pathlib import Path
import json, hashlib, io, shutil, random, collections, warnings, datetime, re, pickle
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageOps
from concurrent.futures import ThreadPoolExecutor
import yaml
warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/WeaveGuard/data')
SRC = ROOT / 'textile_defects'
AUDIT = ROOT.parent / 'audit_output'
DRIVE_OUT = ROOT.parent / 'datasets'
DS = Path('/content/weaveguard_dataset_v1')
DATASET_VERSION = 'v1'

SEED = 42
N_FOLDS = 20
FOLDS = {'train': 14, 'val': 3, 'test': 3}      # 70 % / 15 % / 15 %
SPLITS = ('train', 'val', 'test')
NEAR_T_OVERRIDE = None        # None = choix automatique du seuil de quasi-doublons
MAX_COMP_FRAC = 0.10          # un groupe fusionné ne doit pas dépasser 10 % des images
N_SEEDS_SEARCH = 300
MIN_IMG_PER_CLASS_EVAL = 8    # minimum d'images par classe en val et en test
CROP = 512
CROPS_PER_IMAGE = {'train': 4, 'val': 3, 'test': 3}
N_THREADS = 32
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('SRC existe :', SRC.exists())

In [ ]:
# 3) Inventaire de textile_defects : labels, hash, résolution
NAMES_DEFAULT = ['DropStitch', 'Hole', 'Horizontal Mark', 'Knot', 'Oil Spot', 'Setup', 'Slub', 'Vertical LineMark', 'Yurn Dust']
def load_names():
    for y in list(SRC.glob('*.yaml')) + list(SRC.glob('*.yml')):
        try:
            d = yaml.safe_load(open(y, encoding='utf-8')); n = d.get('names')
            if isinstance(n, dict): n = [n[k] for k in sorted(n, key=lambda k: int(k))]
            if n: return list(n)
        except Exception: pass
    return NAMES_DEFAULT
NAMES = load_names(); NC = len(NAMES)
print('Classes :', NAMES)

img_root = SRC / 'images'
assert img_root.exists(), f'{img_root} introuvable : vérifie SRC'
imgs = sorted(p for p in img_root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT)
print(len(imgs), 'images trouvées')

def dhash(im):
    g = np.asarray(im.convert('L').resize((9, 8)), dtype=np.int16)
    v = 0
    for b in (g[:, 1:] > g[:, :-1]).flatten(): v = (v << 1) | int(b)
    return v

# cache de l'audit (évite de relire Drive) : md5, dhash, taille
cache = {}
cp = AUDIT / 'scan_cache.pkl'
if cp.exists():
    for r in pickle.load(open(cp, 'rb'))[0]:
        if r['dataset'] == 'textile_defects' and not r['corrupt'] and r['dhash'] is not None:
            cache[Path(r['path']).name] = r
    print('Cache audit :', len(cache), 'entrées')

def inspect(p):
    rel = p.relative_to(img_root)
    lp = (SRC / 'labels' / rel).with_suffix('.txt')
    out = dict(path=p, name=p.name, orig_split=rel.parts[0] if len(rel.parts) > 1 else 'unknown',
               boxes=[], bad_lines=0, md5=None, dhash=None, w=None, h=None, error=None)
    c = cache.get(p.name)
    try:
        if c:
            out.update(md5=c['md5'], dhash=c['dhash'], w=c['w'], h=c['h'])
        else:
            data = p.read_bytes(); out['md5'] = hashlib.md5(data).hexdigest()
            with Image.open(io.BytesIO(data)) as im:
                out['w'], out['h'] = im.size; out['dhash'] = dhash(im)
    except Exception as e:
        out['error'] = str(e)
    if lp.exists():
        for line in open(lp, encoding='utf-8', errors='replace').read().splitlines():
            s = line.split()
            if not s: continue
            try:
                cl = int(float(s[0])); v = [float(t) for t in s[1:]]
                if len(v) == 4: x, y, w, h = v
                elif len(v) >= 6 and len(v) % 2 == 0:
                    xs, ys = v[0::2], v[1::2]
                    x, y, w, h = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
                else: raise ValueError
                if not (0 <= cl < NC and 0 < w <= 1 and 0 < h <= 1 and 0 <= x <= 1 and 0 <= y <= 1): raise ValueError
                out['boxes'].append((cl, x, y, w, h))
            except Exception:
                out['bad_lines'] += 1
    return out

with ThreadPoolExecutor(N_THREADS) as ex:
    rows = list(ex.map(inspect, imgs))
df_all = pd.DataFrame(rows)
df_all['n_boxes'] = df_all.boxes.map(len)
df_all['classes'] = df_all.boxes.map(lambda b: sorted(set(x[0] for x in b)))
print('Lignes de labels invalides :', int(df_all.bad_lines.sum()), '| erreurs de lecture image :', int(df_all.error.notna().sum()))

unlabeled = df_all[(df_all.n_boxes == 0) | df_all.error.notna() | df_all.dhash.isna()]
print('Images exclues (sans boîte valide ou illisibles) :', len(unlabeled)); print(list(unlabeled.name)[:10])
df = df_all.drop(unlabeled.index).reset_index(drop=True)
assert df.name.is_unique, 'noms de fichiers non uniques entre partitions'
print('Images retenues :', len(df))

In [ ]:
# 4) Groupes anti-fuite : photo source + doublons exacts + quasi-doublons (dhash)
def photo_id(name):
    m = re.match(r'^(IMG_\d+|PXL_\d{8}_\d+)', name)
    return m.group(1) if m else name.split('.rf.')[0]
df['photo_id'] = df.name.map(photo_id)
n = len(df)
print(f'{df.photo_id.nunique()} photos sources pour {n} images')

arr = np.array([int(v) for v in df.dhash], dtype=np.uint64)
tab = np.array([bin(i).count('1') for i in range(65536)], dtype=np.uint8)
TMAX = 6
pairs = []
for i in range(n - 1):
    x = arr[i+1:] ^ arr[i]
    d = tab[x.view(np.uint16).reshape(-1, 4)].sum(axis=1)
    for j in np.where(d <= TMAX)[0]:
        pairs.append((i, i + 1 + int(j), int(d[j])))

def components(T):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for key in (df.photo_id, df.md5):
        first = {}
        for i, k in enumerate(key):
            if k in first: union(i, first[k])
            else: first[k] = i
    for i, j, d in pairs:
        if d <= T: union(i, j)
    return np.array([find(i) for i in range(n)])

rows = []
for T in range(-1, TMAX + 1):
    sizes = pd.Series(components(T)).value_counts()
    rows.append(dict(seuil_hamming=T, n_groupes=len(sizes), plus_gros_groupe=int(sizes.iloc[0]), part=round(sizes.iloc[0] / n, 3)))
tbl = pd.DataFrame(rows)
print('(seuil -1 = uniquement photo source + doublons exacts)')
display(tbl)
okT = tbl[tbl.part <= MAX_COMP_FRAC]
NEAR_T = NEAR_T_OVERRIDE if NEAR_T_OVERRIDE is not None else (int(okT.seuil_hamming.max()) if len(okT) else -1)
df['group'] = components(NEAR_T)
print(f'>>> Seuil retenu : {NEAR_T} -> {df.group.nunique()} groupes')
if df.group.nunique() < 3 * N_FOLDS:
    print('[!] Peu de groupes : le split sera moins équilibré. Essaie NEAR_T_OVERRIDE plus bas.')

In [ ]:
# 5) Split stratifié par groupes (recherche sur plusieurs graines, score déterministe)
from sklearn.model_selection import StratifiedGroupKFold
P = np.zeros((n, NC), dtype=int)
for i, cl in enumerate(df.classes):
    for c in cl: P[i, c] = 1
cls_freq = P.sum(0); tot = cls_freq.astype(float)
gcls = df.groupby('group').classes.apply(lambda s: set().union(*[set(x) for x in s]))
df['strat'] = df.group.map(lambda g: min(gcls[g], key=lambda c: cls_freq[c]))
target = np.array([FOLDS[s] / N_FOLDS for s in SPLITS])
order = [s for s in SPLITS for _ in range(FOLDS[s])]

best = None
for seed in range(SEED, SEED + N_SEEDS_SEARCH):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold = np.zeros(n, dtype=int)
    for k, (_, te) in enumerate(sgkf.split(np.zeros(n), df.strat.values, df.group.values)):
        fold[te] = k
    perm = np.random.RandomState(seed).permutation(N_FOLDS)
    fold_split = {int(perm[i]): order[i] for i in range(N_FOLDS)}
    sp = np.array([fold_split[f] for f in fold])
    cnt = np.zeros((NC, 3)); score = 0.0
    for si, s in enumerate(SPLITS):
        cnt[:, si] = P[sp == s].sum(0)
        score += 2 * abs((sp == s).mean() - target[si])
    score += np.abs(cnt / tot[:, None] - target[None, :]).sum()
    score += 10 * int(((cnt[:, 1] < MIN_IMG_PER_CLASS_EVAL) | (cnt[:, 2] < MIN_IMG_PER_CLASS_EVAL)).sum())
    if best is None or score < best[0]: best = (score, seed, sp)
score, BEST_SEED, sp = best
df['split'] = sp
print(f'Meilleure graine : {BEST_SEED} (score {score:.3f}, plus petit = mieux)')

In [ ]:
# 6) Vérifications : fuite, équilibre des classes
import matplotlib.pyplot as plt
assert df.groupby('group').split.nunique().max() == 1, 'un groupe traverse deux partitions !'
assert df.groupby('md5').split.nunique().max() == 1, 'doublon exact entre partitions !'
resid = {}
for T in range(NEAR_T + 1, TMAX + 1):
    resid[T] = sum(1 for i, j, d in pairs if d == T and df.split[i] != df.split[j])
cross_at_or_below = sum(1 for i, j, d in pairs if d <= NEAR_T and df.split[i] != df.split[j])
print(f'OK : 0 groupe partagé, 0 doublon exact entre partitions, paires quasi-identiques (<= {NEAR_T}) entre partitions : {cross_at_or_below}')
print('Paires plus lointaines (informatif) entre partitions, par distance :', resid)

print('\nImages par partition :'); print(df.split.value_counts().reindex(SPLITS).to_string())
img_cls = pd.DataFrame({s: P[(df.split == s).values].sum(0) for s in SPLITS}, index=NAMES)
img_cls['total'] = img_cls.sum(axis=1)
print('\nImages contenant la classe (une image peut avoir plusieurs classes) :'); display(img_cls)
box_rows = [dict(split=r.split, cls=NAMES[b[0]]) for r in df.itertuples() for b in r.boxes]
box_cls = pd.DataFrame(box_rows).groupby(['cls', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPLITS), fill_value=0)
box_cls['total'] = box_cls.sum(axis=1)
print('Boîtes par classe :'); display(box_cls)
low = img_cls[(img_cls.val < MIN_IMG_PER_CLASS_EVAL) | (img_cls.test < MIN_IMG_PER_CLASS_EVAL)]
if len(low): print('[!] Classes avec peu d images en val/test :', list(low.index))
img_cls[list(SPLITS)].plot(kind='bar', stacked=True, figsize=(9, 4), title='Images par classe et partition')
plt.tight_layout(); plt.savefig(DRIVE_OUT / 'split_classes_v1.png', dpi=120); plt.show()

In [ ]:
# 7) Export YOLO local (images + labels réécrits en format détection propre) + data.yaml
if DS.exists(): shutil.rmtree(DS)
for s in SPLITS:
    (DS / 'images' / s).mkdir(parents=True); (DS / 'labels' / s).mkdir(parents=True)

def export(r):
    s = r['split']
    shutil.copy2(r['path'], DS / 'images' / s / r['name'])
    lines = [f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}' for c, x, y, w, h in r['boxes']]
    (DS / 'labels' / s / (Path(r['name']).stem + '.txt')).write_text('\n'.join(lines) + '\n', encoding='utf-8')

recs = df[['path', 'name', 'split', 'boxes']].to_dict('records')
with ThreadPoolExecutor(N_THREADS) as ex: list(ex.map(export, recs))

yaml.safe_dump(dict(path=str(DS), train='images/train', val='images/val', test='images/test',
                    nc=NC, names={i: nm for i, nm in enumerate(NAMES)}),
               open(DS / 'data.yaml', 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)
for s in SPLITS:
    print(s, len(list((DS / 'images' / s).iterdir())), 'images,', len(list((DS / 'labels' / s).iterdir())), 'labels')
print((DS / 'data.yaml').read_text())

In [ ]:
# 8) Contrôle visuel : les boîtes sont-elles bien alignées sur les défauts ?
import matplotlib.pyplot as plt
rng = random.Random(SEED)
sample = rng.sample(list(df[df.split == 'train'].itertuples()), min(6, int((df.split == 'train').sum())))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
palette = ['red', 'lime', 'cyan', 'yellow', 'magenta', 'orange', 'white', 'deepskyblue', 'pink']
for ax, r in zip(axes.flat, sample):
    im = ImageOps.exif_transpose(Image.open(DS / 'images' / r.split / r.name)).convert('RGB')
    W, H = im.size; dr = ImageDraw.Draw(im)
    for c, x, y, w, h in r.boxes:
        dr.rectangle([(x-w/2)*W, (y-h/2)*H, (x+w/2)*W, (y+h/2)*H], outline=palette[c % 9], width=max(3, W // 250))
    im.thumbnail((700, 700)); ax.imshow(im)
    ax.set_title(', '.join(sorted(set(NAMES[b[0]] for b in r.boxes))), fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 9) Recadrages sans défaut pour PatchCore (chaque partition utilise ses propres images)
def make_crops(split):
    out_dir = DS / 'normal_crops' / split; out_dir.mkdir(parents=True, exist_ok=True)
    count = skipped = 0
    for p in sorted((DS / 'images' / split).iterdir()):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
        if min(W, H) < CROP: skipped += 1; continue
        bxs = []
        lab = DS / 'labels' / split / (p.stem + '.txt')
        for line in lab.read_text().splitlines():
            s = line.split()
            if len(s) != 5: continue
            _, cx, cy, bw, bh = int(s[0]), *map(float, s[1:])
            x0, y0, x1, y1 = (cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H
            m = max(32, 0.15 * max(x1 - x0, y1 - y0))
            bxs.append((x0 - m, y0 - m, x1 + m, y1 + m))
        rng = random.Random(f'{SEED}-{p.name}'); got = []; tries = 0
        while len(got) < CROPS_PER_IMAGE[split] and tries < 60:
            tries += 1
            x, y = rng.randint(0, W - CROP), rng.randint(0, H - CROP)
            if any(not (x + CROP <= a or x >= c or y + CROP <= b or y >= d) for a, b, c, d in bxs): continue
            if any(max(0, min(x, gx) + CROP - max(x, gx)) * max(0, min(y, gy) + CROP - max(y, gy)) > 0.3 * CROP * CROP for gx, gy in got): continue
            got.append((x, y))
        for k, (x, y) in enumerate(got):
            im.crop((x, y, x + CROP, y + CROP)).save(out_dir / f'{p.stem}_c{k}.jpg', quality=95)
        count += len(got)
    return count, skipped

crop_stats = {}
for s in SPLITS:
    c, sk = make_crops(s); crop_stats[s] = dict(crops=c, images_too_small=sk)
print('Recadrages sans défaut :', crop_stats)

import matplotlib.pyplot as plt
files = sorted((DS / 'normal_crops' / 'train').glob('*.jpg')); random.Random(SEED).shuffle(files)
fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for ax, f in zip(axes.flat, files[:24]):
    ax.imshow(Image.open(f).resize((200, 200))); ax.axis('off')
plt.suptitle('24 recadrages "sans défaut" (train) : vérifie qu aucun défaut visible ne subsiste'); plt.tight_layout(); plt.show()

In [ ]:
# 10) Manifeste, version, zip sur Drive, rapport à coller dans le chat
man = df[['name', 'split', 'group', 'photo_id', 'md5', 'w', 'h', 'n_boxes']].copy()
man['classes'] = df.classes.map(lambda c: ';'.join(NAMES[i] for i in c))
man.to_csv(DS / 'manifest.csv', index=False)
ds_hash = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()

stats = dict(
    dataset_version=DATASET_VERSION, dataset_hash_sha256=ds_hash, created=datetime.datetime.now().isoformat(timespec='seconds'),
    source='textile_defects (BDFD9 et hole_detection écartés)', seed_split=int(BEST_SEED), near_dup_hamming_threshold=int(NEAR_T),
    n_groups=int(df.group.nunique()), n_images=int(n), images_per_split={s: int((df.split == s).sum()) for s in SPLITS},
    images_per_class=img_cls.to_dict('index'), boxes_per_class=box_cls.to_dict('index'),
    excluded_unlabeled=list(unlabeled.name), label_lines_invalid=int(df_all.bad_lines.sum()),
    leakage_checks=dict(shared_groups=0, shared_exact_duplicates=0, near_pairs_below_threshold_across_splits=int(cross_at_or_below),
                        farther_pairs_across_splits=resid),
    normal_crops=dict(size_px=CROP, per_split=crop_stats, note='extraits hors boîtes annotées (marge >= 32 px) ; défauts non annotés possibles'),
    classes=NAMES)
(DS / 'dataset_stats.json').write_text(json.dumps(stats, indent=1, ensure_ascii=False, default=str), encoding='utf-8')

shutil.copy2(DS / 'dataset_stats.json', DRIVE_OUT / 'dataset_stats_v1.json')
shutil.copy2(DS / 'manifest.csv', DRIVE_OUT / 'manifest_v1.csv')
shutil.copy2(DS / 'data.yaml', DRIVE_OUT / 'data_v1.yaml')
zip_path = shutil.make_archive(str(DRIVE_OUT / f'weaveguard_dataset_{DATASET_VERSION}'), 'zip', DS)
print('Zip sauvegardé sur Drive :', zip_path)
print('=' * 70)
print(json.dumps(stats, indent=1, ensure_ascii=False, default=str)[:9000])
try:
    from google.colab import files; files.download(str(DRIVE_OUT / 'dataset_stats_v1.json'))
except Exception: pass